## Fazer um Model Register e registar uma pipepline no mlflow

In [1]:
import mlflow
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [2]:
root_path = '../data/'

SEED = 42
seed = 42

TARGET_COL = "default.payment.next.month"

In [3]:
df = pd.read_csv(root_path + 'lending_data.csv')

## Definir a diretoria onde as experiências são guardadas

In [4]:
from pathlib import Path

uri = "../../mlruns"

Path(uri).mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(uri)

### Fazer set da experiência "Rumos Bank Experiment"

In [5]:
mlflow.set_experiment("Rumos Bank Experiment")

<Experiment: artifact_location='file:///c:/Users/User/python_programing/OML-trabalho-master/rumos_bank/notebooks/../../mlruns/235151208529047428', creation_time=1745165627065, experiment_id='235151208529047428', last_update_time=1745165627065, lifecycle_stage='active', name='Rumos Bank Experiment', tags={}>

## Criar os datasets

In [6]:
train_set, test_set = train_test_split(df, test_size = 0.2, random_state = seed)

X_train = train_set.drop(['default.payment.next.month'], axis = 'columns')
y_train = train_set['default.payment.next.month']

X_test = test_set.drop(['default.payment.next.month'], axis = 1)
y_test = test_set['default.payment.next.month']

## Criar uma run

In [7]:
run = mlflow.start_run(run_name="Random Forest Run - C0.1 - pipeline")
RUN_ID = run.info.run_uuid
RUN_ID

'56fca935833c4132be2f7c525b49d850'

## Guardar datasets, modelos, artefactos, métricas e parametros da run

In [8]:
# guardarmos o dataset de treino e de teste associado à run
train_dataset = mlflow.data.from_pandas(train_set, targets='default.payment.next.month', name="lending_data")
test_dataset = mlflow.data.from_pandas(test_set, targets='default.payment.next.month', name="lending_data")
mlflow.log_input(train_dataset, context="train")
mlflow.log_input(test_dataset, context="test")

# Guardamos a seed utilizado como parametro
mlflow.log_param("seed", SEED)

c:\Users\User\miniconda3\envs\OML\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


42

## Pipeline e registro do melhor modelo

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("random_forest", RandomForestClassifier(random_state=SEED, n_estimators=300)) #nº estimators é 300 porque vimos no notebook1 que era o melhor parametro para a random forest!
])
rf_pipeline.fit(X_train, y_train)
mlflow.sklearn.log_model(rf_pipeline, artifact_path="rf_pipeline", registered_model_name="random_forest")
rf_pipeline

In [ ]:
params=rf_pipeline.get_params()

modified_params = {}
for k, v in params.items():
    new_key = k.replace("random_forest__", '')
    modified_params[new_key] = v

mlflow.log_params(modified_params)
modified_params

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('random_forest',
   RandomForestClassifier(n_estimators=300, random_state=42))],
 'transform_input': None,
 'verbose': 0,
 'scaler': StandardScaler(),
 'random_forest': RandomForestClassifier(n_estimators=300, random_state=42),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 300,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'warm_start': False}

In [ ]:
y_preds = rf_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_preds)
mlflow.log_metric("accuracy", acc)
acc

0.817

## Terminar a run

In [ ]:
mlflow.end_run()

### Não esquecer de executar: mlflow ui --backend-store-uri ./mlruns no terminal do Visual Studio Code

Neste momento tenho na porta 5000 a correr o meu modelo "champion" - Aliased Versions